In [ ]:
print("Hello, EV Charging Optimization!")

Hello, EV Charging Optimization!


In [ ]:
charging_power = 3       # Power in kW
charging_time = 2        # Time in hours
electricity_price = 8    # Price in rupees per kWh

energy = charging_power * charging_time
cost = energy * electricity_price

print("Energy delivered:", energy, "kWh")
print("Charging cost: Rs.", cost)

Energy delivered: 6 kWh
Charging cost: Rs. 48


In [ ]:
ev1_power = 3
ev2_power = 3
ev3_power = 3

station_limit = 6

total_power = ev1_power + ev2_power + ev3_power

print("Total charging demand:", total_power, "kW")
print("Station capacity:", station_limit, "kW")

if total_power > station_limit:
    print("Warning: Charging demand exceeds station capacity!")
else:
    print("Charging demand is within station capacity.")

Total charging demand: 9 kW
Station capacity: 6 kW


In [ ]:

# Our first EV charging dataset

ev_data = [
    {"name": "EV1", "arrival": 0, "departure": 4, "energy": 6, "power": 3},
    {"name": "EV2", "arrival": 0, "departure": 3, "energy": 3, "power": 3},
    {"name": "EV3", "arrival": 1, "departure": 6, "energy": 6, "power": 3},
    {"name": "EV4", "arrival": 2, "departure": 7, "energy": 6, "power": 3},
    {"name": "EV5", "arrival": 3, "departure": 8, "energy": 9, "power": 3}
]

station_limit = 6

for ev in ev_data:
    charging_hours = ev["energy"] / ev["power"]
    print(ev["name"], "needs", charging_hours, "hours of charging")


EV1 needs 2.0 hours of charging
EV2 needs 1.0 hours of charging
EV3 needs 2.0 hours of charging
EV4 needs 2.0 hours of charging
EV5 needs 3.0 hours of charging


In [ ]:

# Each row represents one EV.
# Each column represents a one-hour time slot.
# 1 = charging, 0 = not charging.

schedule = [
    [1, 1, 0, 0, 0, 0, 0, 0],  # EV1: 2 hours
    [0, 0, 1, 0, 0, 0, 0, 0],  # EV2: 1 hour
    [0, 0, 0, 1, 1, 0, 0, 0],  # EV3: 2 hours
    [0, 0, 0, 0, 0, 1, 1, 0],  # EV4: 2 hours
    [0, 0, 0, 0, 0, 0, 0, 0],  # EV5: 0 hours
]

ev_names = ["EV1", "EV2", "EV3", "EV4", "EV5"]
charging_power = 3
station_limit = 6

for t in range(8):
    active_evs = 0

    for i in range(5):
        active_evs += schedule[i][t]

    total_power = active_evs * charging_power

    print("Slot", t, "| EVs charging:", active_evs,
          "| Total power:", total_power, "kW")


Slot 0 | EVs charging: 1 | Total power: 3 kW
Slot 1 | EVs charging: 1 | Total power: 3 kW
Slot 2 | EVs charging: 1 | Total power: 3 kW
Slot 3 | EVs charging: 1 | Total power: 3 kW
Slot 4 | EVs charging: 1 | Total power: 3 kW
Slot 5 | EVs charging: 1 | Total power: 3 kW
Slot 6 | EVs charging: 1 | Total power: 3 kW
Slot 7 | EVs charging: 0 | Total power: 0 kW


In [ ]:

# Maximum power the charging station can supply
station_limit = 6

# Check whether our schedule follows all the rules
for i in range(len(ev_data)):
    ev = ev_data[i]
    row = schedule[i]

    # Calculate how many hours the EV charged
    charged_hours = sum(row)

    # Calculate the energy delivered
    energy_delivered = charged_hours * ev["power"]

    print("\nChecking", ev["name"])
    print("Required energy:", ev["energy"], "kWh")
    print("Energy delivered:", energy_delivered, "kWh")

    # Check 1: Energy requirement
    if energy_delivered >= ev["energy"]:
        print("Energy requirement: PASS")
    else:
        print("Energy requirement: FAIL")

    # Check 2: Arrival and departure times
    valid_time = True

    for t in range(len(row)):
        if row[t] == 1:
            if t < ev["arrival"] or t >= ev["departure"]:
                valid_time = False

    if valid_time:
        print("Availability window: PASS")
    else:
        print("Availability window: FAIL")


# Check 3: Charging station capacity
print("\nChecking station capacity")

for t in range(8):
    total_power = 0

    for i in range(len(ev_data)):
        total_power += schedule[i][t] * ev_data[i]["power"]

    print("Time slot", t, "- Total power:", total_power, "kW")

    if total_power <= station_limit:
        print("Capacity: PASS")
    else:
        print("Capacity: FAIL")



Checking EV1
Required energy: 6 kWh
Energy delivered: 6 kWh
Energy requirement: PASS
Availability window: PASS

Checking EV2
Required energy: 3 kWh
Energy delivered: 3 kWh
Energy requirement: PASS
Availability window: PASS

Checking EV3
Required energy: 6 kWh
Energy delivered: 6 kWh
Energy requirement: PASS
Availability window: PASS

Checking EV4
Required energy: 6 kWh
Energy delivered: 6 kWh
Energy requirement: PASS
Availability window: PASS

Checking EV5
Required energy: 9 kWh
Energy delivered: 0 kWh
Energy requirement: FAIL
Availability window: PASS

Checking station capacity
Time slot 0 - Total power: 3 kW
Capacity: PASS
Time slot 1 - Total power: 3 kW
Capacity: PASS
Time slot 2 - Total power: 3 kW
Capacity: PASS
Time slot 3 - Total power: 3 kW
Capacity: PASS
Time slot 4 - Total power: 3 kW
Capacity: PASS
Time slot 5 - Total power: 3 kW
Capacity: PASS
Time slot 6 - Total power: 3 kW
Capacity: PASS
Time slot 7 - Total power: 0 kW
Capacity: PASS


In [ ]:

# Maximum power of the charging station
station_limit = 6

# Number of one-hour time slots
num_slots = 8

# Start with an empty schedule: all EVs are switched off
schedule_auto = [
    [0] * num_slots for ev in ev_data
]

# Track the energy still required by each EV
remaining_energy = [
    ev["energy"] for ev in ev_data
]

# Process each time slot
for t in range(num_slots):

    # Find EVs that have arrived and have not departed
    available_evs = []

    for i in range(len(ev_data)):
        ev = ev_data[i]

        if ev["arrival"] <= t < ev["departure"]:
            if remaining_energy[i] > 0:
                available_evs.append(i)

    # Prioritize EVs with the earliest departure
    available_evs.sort(
        key=lambda i: ev_data[i]["departure"]
    )

    # Assign charging slots without exceeding station capacity
    power_used = 0

    for i in available_evs:
        ev_power = ev_data[i]["power"]

        if power_used + ev_power <= station_limit:
            schedule_auto[i][t] = 1
            power_used += ev_power
            remaining_energy[i] -= ev_power

    print(
        "Slot", t,
        "| Power used:", power_used, "kW"
    )

# Display the automatically generated schedule
print("\nAutomatically generated schedule:")

for i in range(len(ev_data)):
    print(ev_data[i]["name"], schedule_auto[i])

# Check whether every EV received enough energy
print("\nFinal energy check:")

for i in range(len(ev_data)):
    delivered = sum(schedule_auto[i]) * ev_data[i]["power"]

    print(
        ev_data[i]["name"],
        "| Required:", ev_data[i]["energy"], "kWh",
        "| Delivered:", delivered, "kWh"
    )


Slot 0 | Power used: 6 kW
Slot 1 | Power used: 6 kW
Slot 2 | Power used: 6 kW
Slot 3 | Power used: 6 kW
Slot 4 | Power used: 3 kW
Slot 5 | Power used: 3 kW
Slot 6 | Power used: 0 kW
Slot 7 | Power used: 0 kW

Automatically generated schedule:
EV1 [1, 1, 0, 0, 0, 0, 0, 0]
EV2 [1, 0, 0, 0, 0, 0, 0, 0]
EV3 [0, 1, 1, 0, 0, 0, 0, 0]
EV4 [0, 0, 1, 1, 0, 0, 0, 0]
EV5 [0, 0, 0, 1, 1, 1, 0, 0]

Final energy check:
EV1 | Required: 6 kWh | Delivered: 6 kWh
EV2 | Required: 3 kWh | Delivered: 3 kWh
EV3 | Required: 6 kWh | Delivered: 6 kWh
EV4 | Required: 6 kWh | Delivered: 6 kWh
EV5 | Required: 9 kWh | Delivered: 9 kWh


In [ ]:

# Illustrative electricity price for each hour (₹ per kWh)
electricity_price = [
    12, 12, 10, 10, 6, 5, 5, 5
]

# Calculate the total cost of our current schedule
total_cost = 0

for t in range(8):
    power_used = 0

    for i in range(len(ev_data)):
        power_used += (
            schedule_auto[i][t] * ev_data[i]["power"]
        )

    # Each slot lasts one hour, so energy = power × 1 hour
    energy_used = power_used * 1

    slot_cost = energy_used * electricity_price[t]
    total_cost += slot_cost

    print(
        "Slot", t,
        "| Power:", power_used, "kW",
        "| Price:", electricity_price[t], "₹/kWh",
        "| Cost: ₹", slot_cost
    )

print("\nTotal charging cost: ₹", total_cost)


Slot 0 | Power: 6 kW | Price: 12 ₹/kWh | Cost: ₹ 72
Slot 1 | Power: 6 kW | Price: 12 ₹/kWh | Cost: ₹ 72
Slot 2 | Power: 6 kW | Price: 10 ₹/kWh | Cost: ₹ 60
Slot 3 | Power: 6 kW | Price: 10 ₹/kWh | Cost: ₹ 60
Slot 4 | Power: 3 kW | Price: 6 ₹/kWh | Cost: ₹ 18
Slot 5 | Power: 3 kW | Price: 5 ₹/kWh | Cost: ₹ 15
Slot 6 | Power: 0 kW | Price: 5 ₹/kWh | Cost: ₹ 0
Slot 7 | Power: 0 kW | Price: 5 ₹/kWh | Cost: ₹ 0

Total charging cost: ₹ 297


In [ ]:

# Start with an empty schedule
schedule_cheap = [
    [0] * 8 for ev in ev_data
]

# Schedule each EV in its cheapest available hours
for i, ev in enumerate(ev_data):
    slots_needed = ev["energy"] // ev["power"]

    available_slots = list(
        range(ev["arrival"], ev["departure"])
    )

    # Sort by electricity price: cheapest first
    available_slots.sort(
        key=lambda t: electricity_price[t]
    )

    chosen_slots = available_slots[:slots_needed]

    for t in chosen_slots:
        schedule_cheap[i][t] = 1

# Check total station power
valid_capacity = True

for t in range(8):
    power_used = sum(
        schedule_cheap[i][t] * ev_data[i]["power"]
        for i in range(len(ev_data))
    )

    print("Slot", t, "| Power used:", power_used, "kW")

    if power_used > station_limit:
        valid_capacity = False

# Calculate total cost
total_cost_cheap = 0

for t in range(8):
    power_used = sum(
        schedule_cheap[i][t] * ev_data[i]["power"]
        for i in range(len(ev_data))
    )

    total_cost_cheap += power_used * electricity_price[t]

print("\nCapacity valid:", valid_capacity)
print("New schedule:")

for i, ev in enumerate(ev_data):
    print(ev["name"], schedule_cheap[i])

print("New total cost: ₹", total_cost_cheap)


Slot 0 | Power used: 0 kW
Slot 1 | Power used: 0 kW
Slot 2 | Power used: 6 kW
Slot 3 | Power used: 3 kW
Slot 4 | Power used: 3 kW
Slot 5 | Power used: 9 kW
Slot 6 | Power used: 6 kW
Slot 7 | Power used: 3 kW

Capacity valid: False
New schedule:
EV1 [0, 0, 1, 1, 0, 0, 0, 0]
EV2 [0, 0, 1, 0, 0, 0, 0, 0]
EV3 [0, 0, 0, 0, 1, 1, 0, 0]
EV4 [0, 0, 0, 0, 0, 1, 1, 0]
EV5 [0, 0, 0, 0, 0, 1, 1, 1]
New total cost: ₹ 198


In [ ]:

# Start with an empty schedule
schedule_coordinated = [
    [0] * 8 for ev in ev_data
]

remaining = [ev["energy"] for ev in ev_data]

# Consider the cheapest time slots first
slots_by_price = sorted(
    range(8),
    key=lambda t: electricity_price[t]
)

for t in slots_by_price:

    # Find EVs that can charge during this slot
    candidates = []

    for i, ev in enumerate(ev_data):
        if ev["arrival"] <= t < ev["departure"]:
            if remaining[i] > 0:
                candidates.append(i)

    # Prioritize EVs that depart earlier
    candidates.sort(
        key=lambda i: ev_data[i]["departure"]
    )

    power_used = 0

    for i in candidates:
        power = ev_data[i]["power"]

        if power_used + power <= station_limit:
            schedule_coordinated[i][t] = 1
            power_used += power
            remaining[i] -= power

# Calculate cost and check capacity
total_cost = 0
capacity_valid = True

for t in range(8):
    power_used = sum(
        schedule_coordinated[i][t] * ev_data[i]["power"]
        for i in range(len(ev_data))
    )

    slot_cost = power_used * electricity_price[t]
    total_cost += slot_cost

    if power_used > station_limit:
        capacity_valid = False

    print(
        "Slot", t,
        "| Power:", power_used, "kW",
        "| Cost: ₹", slot_cost
    )

print("\nCoordinated schedule:")
for i, ev in enumerate(ev_data):
    print(ev["name"], schedule_coordinated[i])

print("\nEnergy remaining:", remaining)
print("Capacity valid:", capacity_valid)
print("Total cost: ₹", total_cost)


Slot 0 | Power: 0 kW | Cost: ₹ 0
Slot 1 | Power: 0 kW | Cost: ₹ 0
Slot 2 | Power: 6 kW | Cost: ₹ 60
Slot 3 | Power: 3 kW | Cost: ₹ 30
Slot 4 | Power: 6 kW | Cost: ₹ 36
Slot 5 | Power: 6 kW | Cost: ₹ 30
Slot 6 | Power: 6 kW | Cost: ₹ 30
Slot 7 | Power: 3 kW | Cost: ₹ 15

Coordinated schedule:
EV1 [0, 0, 1, 1, 0, 0, 0, 0]
EV2 [0, 0, 1, 0, 0, 0, 0, 0]
EV3 [0, 0, 0, 0, 1, 1, 0, 0]
EV4 [0, 0, 0, 0, 0, 1, 1, 0]
EV5 [0, 0, 0, 0, 1, 0, 1, 1]

Energy remaining: [0, 0, 0, 0, 0]
Capacity valid: True
Total cost: ₹ 201


In [ ]:

import numpy as np
from scipy.optimize import milp, Bounds, LinearConstraint

n_evs = len(ev_data)
n_slots = len(electricity_price)
n_vars = n_evs * n_slots

# Variable x[i,t] is stored at position i*n_slots + t
def idx(i, t):
    return i * n_slots + t

# Cost of charging each EV in each hour
cost = np.array([
    ev["power"] * electricity_price[t]
    for ev in ev_data
    for t in range(n_slots)
], dtype=float)

# Every decision variable must be either 0 or 1
bounds = Bounds(
    np.zeros(n_vars),
    np.ones(n_vars)
)

integrality = np.ones(n_vars)

constraints_matrix = []
lower_bounds = []
upper_bounds = []

# Constraint 1: Each EV must receive its required energy
for i, ev in enumerate(ev_data):
    row = np.zeros(n_vars)

    for t in range(n_slots):
        row[idx(i, t)] = ev["power"]

    constraints_matrix.append(row)
    lower_bounds.append(ev["energy"])
    upper_bounds.append(np.inf)

# Constraint 2: No charging outside arrival/departure window
for i, ev in enumerate(ev_data):
    for t in range(n_slots):
        if not (ev["arrival"] <= t < ev["departure"]):
            row = np.zeros(n_vars)
            row[idx(i, t)] = 1

            constraints_matrix.append(row)
            lower_bounds.append(0)
            upper_bounds.append(0)

# Constraint 3: Station power must not exceed 6 kW
for t in range(n_slots):
    row = np.zeros(n_vars)

    for i, ev in enumerate(ev_data):
        row[idx(i, t)] = ev["power"]

    constraints_matrix.append(row)
    lower_bounds.append(0)
    upper_bounds.append(6)

constraints = LinearConstraint(
    np.array(constraints_matrix),
    np.array(lower_bounds),
    np.array(upper_bounds)
)

# Solve the minimum-cost problem
result = milp(
    c=cost,
    integrality=integrality,
    bounds=bounds,
    constraints=constraints
)

if result.success:
    optimal_schedule = np.rint(
        result.x
    ).astype(int).reshape(n_evs, n_slots)

    print("Optimal cost: ₹", round(result.fun, 2))
    print("\nOptimal charging schedule:")

    for i, ev in enumerate(ev_data):
        print(ev["name"], optimal_schedule[i].tolist())

    print("\nStation power by hour:")

    for t in range(n_slots):
        power = sum(
            optimal_schedule[i, t] * ev_data[i]["power"]
            for i in range(n_evs)
        )
        print("Slot", t, ":", power, "kW")

else:
    print("Solver could not find a solution:", result.message)


Optimal cost: ₹ 201.0

Optimal charging schedule:
EV1 [0, 0, 1, 1, 0, 0, 0, 0]
EV2 [0, 0, 1, 0, 0, 0, 0, 0]
EV3 [0, 0, 0, 0, 1, 1, 0, 0]
EV4 [0, 0, 0, 0, 0, 1, 1, 0]
EV5 [0, 0, 0, 0, 1, 0, 1, 1]

Station power by hour:
Slot 0 : 0 kW
Slot 1 : 0 kW
Slot 2 : 6 kW
Slot 3 : 3 kW
Slot 4 : 6 kW
Slot 5 : 6 kW
Slot 6 : 6 kW
Slot 7 : 3 kW


In [ ]:

# Number of EVs and time slots
n_evs = len(ev_data)
n_slots = len(electricity_price)

# Electricity cost for each EV in each time slot
cost_matrix = []

for i in range(n_evs):
    row = []

    for t in range(n_slots):
        cost = ev_data[i]["power"] * electricity_price[t]
        row.append(cost)

    cost_matrix.append(row)

print("Charging cost matrix (₹):")

for i in range(n_evs):
    print(ev_data[i]["name"], cost_matrix[i])


Charging cost matrix (₹):
EV1 [36, 36, 30, 30, 18, 15, 15, 15]
EV2 [36, 36, 30, 30, 18, 15, 15, 15]
EV3 [36, 36, 30, 30, 18, 15, 15, 15]
EV4 [36, 36, 30, 30, 18, 15, 15, 15]
EV5 [36, 36, 30, 30, 18, 15, 15, 15]


In [ ]:

# Penalty strength
penalty_strength = 100

# Calculate the number of hours each EV needs
for ev in ev_data:
    required_hours = ev["energy"] / ev["power"]

    print(
        ev["name"],
        "needs",
        int(required_hours),
        "charging hours"
    )

# Demonstrate the penalty for EV1
required_hours = 2

# Example A: EV1 charges for 2 hours
actual_hours = 2
penalty_a = penalty_strength * (actual_hours - required_hours) ** 2

# Example B: EV1 charges for only 1 hour
actual_hours = 1
penalty_b = penalty_strength * (actual_hours - required_hours) ** 2

print("\nPenalty when EV1 charges for 2 hours:", penalty_a)
print("Penalty when EV1 charges for 1 hour:", penalty_b)


EV1 needs 2 charging hours
EV2 needs 1 charging hours
EV3 needs 2 charging hours
EV4 needs 2 charging hours
EV5 needs 3 charging hours

Penalty when EV1 charges for 2 hours: 0
Penalty when EV1 charges for 1 hour: 100


In [ ]:

import numpy as np

# Create variables only for valid charging hours
variables = []

for i, ev in enumerate(ev_data):
    for t in range(ev["arrival"], ev["departure"]):
        variables.append((i, t))

# Add two binary slack variables per hour.
# These allow unused station capacity to be represented.
slack_variables = []

for t in range(len(electricity_price)):
    slack_variables.append(("slack1", t))
    slack_variables.append(("slack2", t))

all_variables = variables + slack_variables
n = len(all_variables)

# Map each variable to a matrix index
var_index = {
    variable: j for j, variable in enumerate(all_variables)
}

# QUBO objective: x.T @ Q @ x
Q = np.zeros((n, n))

def add_square(coefficients, target, strength):
    """
    Add strength * (sum(coefficients * variables) - target)^2
    to the QUBO matrix.
    """
    for a, ca in coefficients.items():
        ia = var_index[a]

        # Linear term, using x*x = x for binary variables
        Q[ia, ia] += strength * (ca**2 - 2 * target * ca)

        # Quadratic terms
        for b, cb in coefficients.items():
            ib = var_index[b]

            if ia < ib:
                Q[ia, ib] += 2 * strength * ca * cb

# 1. Electricity cost
for i, t in variables:
    v = (i, t)
    Q[var_index[v], var_index[v]] += (
        ev_data[i]["power"] * electricity_price[t]
    )

# 2. Each EV must charge for exactly its required number of hours
energy_penalty = 100

for i, ev in enumerate(ev_data):
    coefficients = {
        (i, t): 1
        for t in range(ev["arrival"], ev["departure"])
    }

    required_hours = int(ev["energy"] / ev["power"])

    add_square(
        coefficients,
        required_hours,
        energy_penalty
    )

# 3. Station capacity: at most 2 EVs per hour
capacity_penalty = 100

for t in range(len(electricity_price)):
    coefficients = {
        (i, t): 1
        for i, ev in enumerate(ev_data)
        if ev["arrival"] <= t < ev["departure"]
    }

    # Slack = slack1 + 2*slack2, allowing values 0, 1, 2, 3.
    coefficients[("slack1", t)] = 1
    coefficients[("slack2", t)] = 2

    add_square(
        coefficients,
        2,
        capacity_penalty
    )

print("Number of charging variables:", len(variables))
print("Number of slack variables:", len(slack_variables))
print("Total QUBO variables:", n)
print("QUBO matrix shape:", Q.shape)
print("QUBO matrix constructed successfully!")


Number of charging variables: 22
Number of slack variables: 16
Total QUBO variables: 38
QUBO matrix shape: (38, 38)
QUBO matrix constructed successfully!


In [ ]:

import numpy as np
import random
import math

random.seed(42)
np.random.seed(42)

# Calculate the QUBO score.
# Q stores diagonal terms and upper-triangle interactions.
def qubo_score(x):
    score = 0.0

    for i in range(len(x)):
        score += Q[i, i] * x[i]

        for j in range(i + 1, len(x)):
            score += Q[i, j] * x[i] * x[j]

    return score


# Simulated annealing search
best_x = None
best_score = float("inf")

number_of_restarts = 20
steps_per_restart = 20000

for restart in range(number_of_restarts):

    # Start with a random binary configuration
    x = np.random.randint(0, 2, size=n)
    current_score = qubo_score(x)

    for step in range(steps_per_restart):

        # Gradually lower the temperature
        progress = step / steps_per_restart
        temperature = 30 * (0.1 / 30) ** progress

        # Try flipping one binary variable
        k = random.randrange(n)
        candidate = x.copy()
        candidate[k] = 1 - candidate[k]

        candidate_score = qubo_score(candidate)
        difference = candidate_score - current_score

        # Accept improvements; sometimes accept worse moves
        if difference < 0 or random.random() < math.exp(
            -difference / temperature
        ):
            x = candidate
            current_score = candidate_score

        if current_score < best_score:
            best_x = x.copy()
            best_score = current_score

print("Best QUBO objective score:", best_score)
print("Simulated annealing search completed.")


Best QUBO objective score: -5193.0
Simulated annealing search completed.


In [ ]:

# Extract the EV charging decisions from the best QUBO solution
qubo_schedule = np.zeros((n_evs, n_slots), dtype=int)

for i, t in variables:
    qubo_schedule[i, t] = best_x[var_index[(i, t)]]

print("QUBO charging schedule:")

for i, ev in enumerate(ev_data):
    print(ev["name"], qubo_schedule[i].tolist())

# Calculate actual electricity cost and validate constraints
actual_cost = 0
valid = True

print("\nValidation:")

for i, ev in enumerate(ev_data):
    delivered = (
        sum(qubo_schedule[i]) * ev["power"]
    )

    print(
        ev["name"],
        "| Required:", ev["energy"], "kWh",
        "| Delivered:", delivered, "kWh"
    )

    if delivered != ev["energy"]:
        valid = False

for t in range(n_slots):
    power = sum(
        qubo_schedule[i, t] * ev_data[i]["power"]
        for i in range(n_evs)
    )

    actual_cost += power * electricity_price[t]

    print("Slot", t, "| Power:", power, "kW")

    if power > station_limit:
        valid = False

print("\nActual electricity cost: ₹", actual_cost)
print("All constraints satisfied:", valid)


QUBO charging schedule:
EV1 [0, 1, 0, 1, 0, 0, 0, 0]
EV2 [0, 0, 1, 0, 0, 0, 0, 0]
EV3 [0, 0, 0, 0, 1, 1, 0, 0]
EV4 [0, 0, 0, 0, 0, 1, 1, 0]
EV5 [0, 0, 0, 0, 1, 0, 1, 1]

Validation:
EV1 | Required: 6 kWh | Delivered: 6 kWh
EV2 | Required: 3 kWh | Delivered: 3 kWh
EV3 | Required: 6 kWh | Delivered: 6 kWh
EV4 | Required: 6 kWh | Delivered: 6 kWh
EV5 | Required: 9 kWh | Delivered: 9 kWh
Slot 0 | Power: 0 kW
Slot 1 | Power: 3 kW
Slot 2 | Power: 3 kW
Slot 3 | Power: 3 kW
Slot 4 | Power: 6 kW
Slot 5 | Power: 6 kW
Slot 6 | Power: 6 kW
Slot 7 | Power: 3 kW

Actual electricity cost: ₹ 207
All constraints satisfied: True


In [ ]:

import numpy as np
import random
import math

def run_annealing(seed, steps=20000):
    random.seed(seed)
    np.random.seed(seed)

    x = np.random.randint(0, 2, size=n)
    score = qubo_score(x)

    best_x = x.copy()
    best_score = score

    for step in range(steps):
        progress = step / steps
        temperature = 30 * (0.1 / 30) ** progress

        k = random.randrange(n)
        candidate = x.copy()
        candidate[k] = 1 - candidate[k]

        candidate_score = qubo_score(candidate)
        difference = candidate_score - score

        if difference < 0 or random.random() < math.exp(
            -difference / temperature
        ):
            x = candidate
            score = candidate_score

        if score < best_score:
            best_x = x.copy()
            best_score = score

    return best_x, best_score


def check_schedule(x):
    schedule = np.zeros((n_evs, n_slots), dtype=int)

    for i, t in variables:
        schedule[i, t] = x[var_index[(i, t)]]

    # Check energy requirements
    for i, ev in enumerate(ev_data):
        delivered = sum(schedule[i]) * ev["power"]
        if delivered != ev["energy"]:
            return False, None, None

    # Check station capacity and calculate actual cost
    total_cost = 0

    for t in range(n_slots):
        power = sum(
            schedule[i, t] * ev_data[i]["power"]
            for i in range(n_evs)
        )

        if power > station_limit:
            return False, None, None

        total_cost += power * electricity_price[t]

    return True, total_cost, schedule


best_valid_cost = float("inf")
best_valid_schedule = None
valid_trials = 0

for seed in range(30):
    x, score = run_annealing(seed)
    valid, cost, schedule = check_schedule(x)

    if valid:
        valid_trials += 1

        if cost < best_valid_cost:
            best_valid_cost = cost
            best_valid_schedule = schedule.copy()

print("Total trials:", 30)
print("Valid trials:", valid_trials)

if best_valid_schedule is not None:
    print("Cheapest valid QUBO cost: ₹", best_valid_cost)
    print("\nBest valid schedule:")

    for i, ev in enumerate(ev_data):
        print(ev["name"], best_valid_schedule[i].tolist())
else:
    print("No valid schedule found in these trials.")


Total trials: 30
Valid trials: 30
Cheapest valid QUBO cost: ₹ 222

Best valid schedule:
EV1 [1, 0, 0, 1, 0, 0, 0, 0]
EV2 [0, 0, 1, 0, 0, 0, 0, 0]
EV3 [0, 0, 0, 1, 1, 0, 0, 0]
EV4 [0, 0, 0, 0, 0, 1, 1, 0]
EV5 [0, 0, 0, 0, 1, 1, 1, 0]


In [ ]:

# Project results collected so far

results = [
    {
        "Method": "Baseline schedule",
        "Cost (Rs)": 297,
        "Feasible": "Yes"
    },
    {
        "Method": "Independent cheap-hour heuristic",
        "Cost (Rs)": 198,
        "Feasible": "No"
    },
    {
        "Method": "Coordinated heuristic",
        "Cost (Rs)": 201,
        "Feasible": "Yes"
    },
    {
        "Method": "Classical MILP",
        "Cost (Rs)": 201,
        "Feasible": "Yes"
    },
    {
        "Method": "QUBO simulated annealing",
        "Cost (Rs)": 207,
        "Feasible": "Yes"
    }
]

import pandas as pd

df = pd.DataFrame(results)

baseline_cost = 297
df["Savings vs baseline (%)"] = (
    (baseline_cost - df["Cost (Rs)"]) / baseline_cost * 100
).round(2)

print(df.to_string(index=False))


                          Method  Cost (Rs) Feasible  Savings vs baseline (%)
               Baseline schedule        297      Yes                     0.00
Independent cheap-hour heuristic        198       No                    33.33
           Coordinated heuristic        201      Yes                    32.32
                  Classical MILP        201      Yes                    32.32
        QUBO simulated annealing        207      Yes                    30.30


In [ ]:

import numpy as np
import random
import math

best_feasible_cost = float("inf")
best_feasible_schedule = None
valid_solutions_found = 0

# Run several independent searches
for seed in range(10):
    random.seed(seed)
    np.random.seed(seed)

    # Start with a random binary configuration
    x = np.random.randint(0, 2, size=n)
    current_score = qubo_score(x)

    # Check the starting configuration
    valid, cost, schedule = check_schedule(x)

    if valid:
        valid_solutions_found += 1

        if cost < best_feasible_cost:
            best_feasible_cost = cost
            best_feasible_schedule = schedule.copy()

    # Simulated annealing
    steps = 10000

    for step in range(steps):
        progress = step / steps
        temperature = 30 * (0.1 / 30) ** progress

        # Flip one binary decision
        k = random.randrange(n)
        candidate = x.copy()
        candidate[k] = 1 - candidate[k]

        candidate_score = qubo_score(candidate)
        difference = candidate_score - current_score

        # Accept or reject the proposed change
        if difference < 0 or random.random() < math.exp(
            -difference / temperature
        ):
            x = candidate
            current_score = candidate_score

        # Check the candidate, even if it was not accepted
        valid, cost, schedule = check_schedule(candidate)

        if valid:
            valid_solutions_found += 1

            if cost < best_feasible_cost:
                best_feasible_cost = cost
                best_feasible_schedule = schedule.copy()

print("Independent searches:", 10)
print("Valid configurations encountered:", valid_solutions_found)

if best_feasible_schedule is not None:
    print("Best feasible electricity cost: ₹", best_feasible_cost)

    print("\nBest feasible schedule:")
    for i, ev in enumerate(ev_data):
        print(ev["name"], best_feasible_schedule[i].tolist())

    print("\nClassical MILP benchmark: ₹201")
    print("Cost difference:", best_feasible_cost - 201, "rupees")
else:
    print("No feasible schedule encountered. Try more search steps.")


Independent searches: 10
Valid configurations encountered: 39774
Best feasible electricity cost: ₹ 216

Best feasible schedule:
EV1 [0, 0, 1, 1, 0, 0, 0, 0]
EV2 [0, 0, 1, 0, 0, 0, 0, 0]
EV3 [0, 0, 0, 0, 1, 1, 0, 0]
EV4 [0, 0, 0, 0, 1, 0, 1, 0]
EV5 [0, 0, 0, 1, 0, 1, 1, 0]

Classical MILP benchmark: ₹201
Cost difference: 15 rupees


In [ ]:

# Test the QUBO score against the direct matrix calculation

def direct_qubo_score(x):
    return float(x @ Q @ x)

def manual_qubo_score(x):
    score = 0.0

    for i in range(len(x)):
        score += Q[i, i] * x[i]

        for j in range(i + 1, len(x)):
            score += Q[i, j] * x[i] * x[j]

    return score

# Create a test vector from the classical optimal schedule
test_x = np.zeros(n, dtype=int)

for i, ev in enumerate(ev_data):
    for t in range(n_slots):
        if optimal_schedule[i, t] == 1:
            test_x[var_index[(i, t)]] = 1

print("Direct matrix score:", direct_qubo_score(test_x))
print("Manual score:", manual_qubo_score(test_x))
print("Scores match:", np.isclose(
    direct_qubo_score(test_x),
    manual_qubo_score(test_x)
))


Direct matrix score: -4199.0
Manual score: -4199.0
Scores match: True


In [ ]:

# Use the known classical optimal schedule
test_x = np.zeros(n, dtype=int)

for i, ev in enumerate(ev_data):
    for t in range(n_slots):
        if optimal_schedule[i, t] == 1:
            test_x[var_index[(i, t)]] = 1

# Set slack variables so each capacity equation is satisfied
for t in range(n_slots):
    active_evs = sum(
        optimal_schedule[i, t]
        for i in range(n_evs)
    )

    unused_capacity = 2 - active_evs

    # Represent unused capacity as slack1 + 2*slack2
    if unused_capacity == 1:
        test_x[var_index[("slack1", t)]] = 1
    elif unused_capacity == 2:
        test_x[var_index[("slack2", t)]] = 1

# Calculate the actual electricity bill
actual_cost = sum(
    optimal_schedule[i, t]
    * ev_data[i]["power"]
    * electricity_price[t]
    for i in range(n_evs)
    for t in range(n_slots)
)

# Calculate energy penalty before adding omitted constants
energy_hours = [
    sum(optimal_schedule[i])
    for i in range(n_evs)
]
required_hours = [
    ev["energy"] // ev["power"]
    for ev in ev_data
]

energy_penalty_value = sum(
    (energy_hours[i] - required_hours[i]) ** 2
    for i in range(n_evs)
) * energy_penalty

# Calculate capacity penalty using the chosen slack values
capacity_penalty_value = 0

for t in range(n_slots):
    active_evs = sum(
        optimal_schedule[i, t]
        for i in range(n_evs)
    )

    slack_value = (
        test_x[var_index[("slack1", t)]]
        + 2 * test_x[var_index[("slack2", t)]]
    )

    capacity_penalty_value += (
        active_evs + slack_value - 2
    ) ** 2

capacity_penalty_value *= capacity_penalty

# Compare the matrix score with the expected score.
# The squared penalty expansion omits constant terms.
omitted_constant = (
    energy_penalty * sum(h ** 2 for h in required_hours)
    + capacity_penalty * n_slots * 2 ** 2
)

matrix_score = qubo_score(test_x)
expected_score = (
    actual_cost
    + energy_penalty_value
    + capacity_penalty_value
    - omitted_constant
)

print("Actual electricity cost: ₹", actual_cost)
print("Energy penalty:", energy_penalty_value)
print("Capacity penalty:", capacity_penalty_value)
print("Matrix score:", matrix_score)
print("Expected matrix score:", expected_score)
print("Formulation check:", np.isclose(matrix_score, expected_score))


Actual electricity cost: ₹ 201
Energy penalty: 0
Capacity penalty: 0
Matrix score: -5199.0
Expected matrix score: -5199
Formulation check: True


In [ ]:
print("Classical MILP cost: ₹201")
print("Best QUBO cost found so far: ₹207")
print("Cost difference: ₹", 207 - 201)
print("QUBO cost above MILP (%):", round((207 - 201) / 201 * 100, 2), "%")

Classical MILP cost: ₹201
Best QUBO cost found so far: ₹207
Cost difference: ₹ 6
QUBO cost above MILP (%): 2.99 %
